# Mini-FORESIGHT — Sales Forecasting

The objective of this step is to **predict future daily sales** for each SKU using the engineered historical sales features.

The model will use:
- `lag_1`
- `lag_2`
- `lag_3`
- `rolling_mean_3`
- `rolling_mean_7`
- `day_of_week`
- `is_weekend`

The target variable is:
- `units_sold`

We use a **simple machine-learning regression model** (Random Forest).

This notebook will:
1. Load the engineered feature dataset.
2. Prepare X and y.
3. Split the data chronologically.
4. Train a regression model.
5. Generate predictions.
6. Display actual vs predicted demand.
7. Produce a future forecast for each SKU.

> **Important**: time-series data must **NOT** be randomly shuffled, because future information must not leak into the training data.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import matplotlib.pyplot as plt

print("Libraries imported successfully")

In [ ]:
features_path = Path("../data/processed/features.csv")

features = pd.read_csv(features_path)

features["date"] = pd.to_datetime(features["date"])

features = features.sort_values(["sku_id", "date"]).reset_index(drop=True)

print("Feature dataset loaded successfully")
print("Shape:", features.shape)
print()
print(features.head())

## Understanding the Forecasting Dataset

Each row represents **one SKU on one date**.

The model will learn the relationship between **historical demand features** and **today's actual sales**.

**Target:**
- `units_sold`

**Input features:**
- `lag_1`
- `lag_2`
- `lag_3`
- `rolling_mean_3`
- `rolling_mean_7`
- `day_of_week`
- `is_weekend`

**Do not use** `date` or `sku_id` as direct numerical model inputs.

SKU-specific forecasting is handled separately so that **each SKU gets its own demand model**.

In [ ]:
feature_columns = [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_3",
    "rolling_mean_7",
    "day_of_week",
    "is_weekend"
]

target_column = "units_sold"

print("Features:")
print(feature_columns)

print()
print("Target:")
print(target_column)

## Chronological Train/Test Split

Because this is a **time-series forecasting** problem, the data must be split **chronologically**.

For each SKU:
- **Earlier** observations are used for **training**.
- The **latest** observations are used for **testing**.

We use approximately:
- **70% training**
- **30% testing**

We do **NOT** use `train_test_split` with `shuffle=True`, because that would randomly mix future rows into the training data.

In [ ]:
# Chronological split per SKU (70% train / 30% test)
train_data = []
test_data = []

for sku in sorted(features["sku_id"].unique()):
    sku_df = features[features["sku_id"] == sku].sort_values("date").reset_index(drop=True)

    split_idx = int(len(sku_df) * 0.7)

    sku_train = sku_df.iloc[:split_idx].copy()
    sku_test = sku_df.iloc[split_idx:].copy()

    train_data.append(sku_train)
    test_data.append(sku_test)

    print(sku)
    print("Total rows:   ", len(sku_df))
    print("Training rows:", len(sku_train))
    print("Testing rows: ", len(sku_test))
    print("Training period:", sku_train["date"].min().date(), "to", sku_train["date"].max().date())
    print("Testing period: ", sku_test["date"].min().date(), "to", sku_test["date"].max().date())
    print()

train_data = pd.concat(train_data, ignore_index=True)
test_data = pd.concat(test_data, ignore_index=True)

In [ ]:
# Train one RandomForestRegressor per SKU
models = {}
predictions_list = []

for sku in sorted(train_data["sku_id"].unique()):
    train_sku = train_data[train_data["sku_id"] == sku]
    test_sku = test_data[test_data["sku_id"] == sku]

    X_train = train_sku[feature_columns]
    y_train = train_sku[target_column]
    X_test = test_sku[feature_columns]
    y_test = test_sku[target_column]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        max_depth=5
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    models[sku] = model

    predictions_list.append(
        pd.DataFrame({
            "date": test_sku["date"].values,
            "sku_id": sku,
            "actual_units_sold": y_test.values,
            "predicted_units_sold": predictions
        })
    )

    print("Trained model for", sku)

predictions_df = pd.concat(predictions_list, ignore_index=True)

In [ ]:
print("=== Forecast Test Results ===")
print()

for sku in sorted(predictions_df["sku_id"].unique()):
    print(sku)
    sku_results = predictions_df[predictions_df["sku_id"] == sku].copy()
    sku_results["predicted_units_sold"] = sku_results["predicted_units_sold"].round(2)
    print(sku_results.to_string(index=False))
    print()

## Model Error — Preliminary Check

MAE and RMSE are calculated here **only as a preliminary model diagnostic**.

The **official** forecast comparison using the **naive baseline** and **WAPE** will be performed in the **next notebook**.

Therefore, we do **NOT** make conclusions here about whether the model is better than a baseline.

In [ ]:
print("=== Preliminary Forecast Metrics ===")
print()

for sku in sorted(predictions_df["sku_id"].unique()):
    sku_results = predictions_df[predictions_df["sku_id"] == sku]

    y_actual = sku_results["actual_units_sold"]
    y_pred = sku_results["predicted_units_sold"]

    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

    print(sku + ":")
    print("MAE:  {:.2f}".format(mae))
    print("RMSE: {:.2f}".format(rmse))
    print()

In [ ]:
# One actual-vs-predicted chart per SKU
for sku in sorted(predictions_df["sku_id"].unique()):
    sku_results = predictions_df[predictions_df["sku_id"] == sku].sort_values("date")

    plt.figure(figsize=(10, 5))
    plt.plot(sku_results["date"], sku_results["actual_units_sold"], marker="o", label="Actual")
    plt.plot(sku_results["date"], sku_results["predicted_units_sold"], marker="s", label="Predicted")
    plt.title("Actual vs Predicted Sales — " + sku)
    plt.xlabel("Date")
    plt.ylabel("Units Sold")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Forecasting Future Demand

The trained models will now be used to estimate demand for the **next 3 days**.

Because the original dataset only contains **14 days** of history and the engineered dataset contains **7 usable days** per SKU after lag/rolling requirements, this project will demonstrate a short **3-day forecast**.

> This is a **demonstration forecasting pipeline**. A production forecasting system would normally require much more historical data.

In [ ]:
future_forecasts = []

forecast_horizon = 3

for sku in sorted(features["sku_id"].unique()):
    sku_hist = features[features["sku_id"] == sku].sort_values("date").reset_index(drop=True)

    model = models[sku]

    latest_date = sku_hist["date"].max()

    # Most recent known demand values (oldest -> newest)
    recent_demand = sku_hist["units_sold"].tail(7).tolist()

    for step in range(1, forecast_horizon + 1):
        future_date = latest_date + pd.Timedelta(days=step)

        # Build the feature vector for this future day
        # recent_demand[-1] is the most recent actual/predicted demand
        lag_1 = recent_demand[-1]
        lag_2 = recent_demand[-2]
        lag_3 = recent_demand[-3]

        rolling_mean_3 = np.mean(recent_demand[-3:])
        rolling_mean_7 = np.mean(recent_demand[-7:])

        day_of_week = future_date.dayofweek
        is_weekend = 1 if day_of_week >= 5 else 0

        X_future = pd.DataFrame([[
            lag_1, lag_2, lag_3,
            rolling_mean_3, rolling_mean_7,
            day_of_week, is_weekend
        ]], columns=feature_columns)

        pred = model.predict(X_future)[0]

        future_forecasts.append({
            "date": future_date,
            "sku_id": sku,
            "forecast_units": pred
        })

        # Use the prediction as the newest demand value for the next step
        recent_demand.append(pred)

future_forecasts = pd.DataFrame(future_forecasts)

future_forecasts = future_forecasts.sort_values(["sku_id", "date"]).reset_index(drop=True)

print("=== 3-Day Demand Forecast ===")
print()
print(future_forecasts.to_string(index=False))

In [ ]:
# Round forecast values for display
display_forecast = future_forecasts.copy()

display_forecast["forecast_units"] = (
    display_forecast["forecast_units"].clip(lower=0).round(2)
)

print("=== Final 3-Day Demand Forecast ===")
print(display_forecast.to_string(index=False))

In [ ]:
output_path = Path("../data/processed/forecast_results.csv")

display_forecast.to_csv(output_path, index=False)

print()
print("Forecast results saved successfully:")
print(output_path)

In [ ]:
# Reload and verify the saved forecast file
verification = pd.read_csv(output_path)

print("=== Saved Forecast Verification ===")
print("Shape:", verification.shape)
print()
print("Columns:", verification.columns.tolist())
print()
print(verification)
print()
print("Missing values:")
print(verification.isna().sum())

## Forecasting Summary

- A **separate Random Forest regression model** was trained for **each SKU**.
- **Historical lag and rolling-demand features** were used as inputs.
- The **train/test split was chronological** to avoid future-data leakage.
- The model generated predictions for the **held-out historical test period**.
- A **3-day future demand forecast** was generated for each SKU.
- The forecast results were saved to:
  - `data/processed/forecast_results.csv`

> **Do not conclude** that the model is accurate or better than a baseline yet.

The next step is:

---

## Step 7 — Baseline Forecast and WAPE Evaluation

That step will compare the machine-learning forecast against a **simple naive forecast** and calculate **WAPE**.